# AutoShotV2 - Eval tren ClipShots test set

Datasets can attach:
- `/kaggle/input/datasets/domanh704/heatmap` - source code autoshotv2
- `/kaggle/input/datasets/domanh704/dataset-clipshots` - ClipShots (annotations/videos)
- `/kaggle/input/datasets/domanh704/result-heatmap1` - checkpoint da train (ckpt_phase2_best.pth)

Eval: chay inference tren ClipShots **test** videos, tinh F1 theo chuan goc AutoShot
(mAP_f1_p_fix_r, sweep threshold lay max F1).


In [ ]:
import os, shutil, subprocess, sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'

r = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                   capture_output=True, text=True, check=False)
print('GPU:', (r.stdout.strip().split(chr(10))[0] if r.returncode==0 else 'none'))
import torch
print(f'torch {torch.__version__}  cuda={torch.cuda.is_available()}')

# ── Paths ─────────────────────────────────────────────────────────────────────
SRC_ROOT  = Path('/kaggle/input/datasets/domanh704/heatmap')
CLIP_BASE = Path('/kaggle/input/datasets/domanh704/dataset-clipshots')

# ── Tim SRC_DIR (autoshotv2 package) ──────────────────────────────────────────
SRC_DIR = None
for init in Path('/kaggle/input').rglob('autoshotv2/__init__.py'):
    for anc in [init.parent.parent.parent] + list(init.parents)[:4]:
        if (anc/'pyproject.toml').exists():
            SRC_DIR = anc; break
    if SRC_DIR: break
if SRC_DIR is None:
    raise FileNotFoundError('Khong tim thay source autoshotv2 (pyproject.toml)')
print('SRC_DIR:', SRC_DIR)

# ── Tim ClipShots root (chua annotations/test.json + videos/) ─────────────────
CLIPSHOTS_ROOT = None
for cand in CLIP_BASE.rglob('annotations/test.json'):
    if (cand.parent.parent/'videos').exists():
        CLIPSHOTS_ROOT = cand.parent.parent; break
if CLIPSHOTS_ROOT is None:
    raise FileNotFoundError('Khong tim thay ClipShots root (annotations/test.json)')
print('CLIPSHOTS_ROOT:', CLIPSHOTS_ROOT)

# ── Tim checkpoint da train ───────────────────────────────────────────────────
CKPT = next(Path('/kaggle/input').rglob('ckpt_phase2_best.pth'), None)
if CKPT is None:
    raise FileNotFoundError('Khong tim thay ckpt_phase2_best.pth (attach result-heatmap1)')
print('CKPT:', CKPT)

WORK = Path('/kaggle/working/clipshots_eval')
WORK.mkdir(parents=True, exist_ok=True)
print('WORK:', WORK)


In [ ]:
# Cai package autoshotv2 (input read-only -> copy sang working)
SRC_WORK = Path('/kaggle/working/autoshotv2_pkg')
if not SRC_WORK.exists():
    shutil.copytree(SRC_DIR, SRC_WORK)
SRC_PY = str(SRC_WORK/'src')
if SRC_PY not in sys.path:
    sys.path.insert(0, SRC_PY)
existing = os.environ.get('PYTHONPATH','')
if SRC_PY not in existing:
    os.environ['PYTHONPATH'] = SRC_PY + (os.pathsep+existing if existing else '')
subprocess.run([sys.executable,'-m','pip','install','-e',str(SRC_WORK),
                '--no-deps','--no-build-isolation','-q'], check=True)
import importlib.util as _ilu
assert _ilu.find_spec('autoshotv2'), 'autoshotv2 khong import duoc'
import autoshotv2
print('autoshotv2 OK:', autoshotv2.__file__)


In [ ]:
# Build GT scenes dict cho ClipShots test (transitions -> scenes)
import json, pickle
import numpy as np

# transitions_to_scenes (nhung inline, copy tu autoshotv2.phase2_data)
def transitions_to_scenes(transitions, n_frames):
    transitions = np.asarray(transitions, dtype=np.int32)
    if n_frames <= 0:
        return np.asarray([[0, 0]], dtype=np.int32)
    if transitions.size == 0:
        return np.asarray([[0, n_frames - 1]], dtype=np.int32)
    transitions = transitions.reshape(-1, 2)
    transitions = transitions[np.argsort(transitions[:, 0])]
    transitions = np.clip(transitions, 0, n_frames - 1)
    transitions = transitions[transitions[:, 0] <= transitions[:, 1]]
    if len(transitions) == 0:
        return np.asarray([[0, n_frames - 1]], dtype=np.int32)
    scenes = [[0, int(transitions[0, 0])]]
    for i in range(1, len(transitions)):
        scenes.append([int(transitions[i - 1, 1]), int(transitions[i, 0])])
    scenes.append([int(transitions[-1, 1]), n_frames - 1])
    arr = np.asarray(scenes, dtype=np.int32)
    arr = np.clip(arr, 0, n_frames - 1)
    arr = arr[arr[:, 0] <= arr[:, 1]]
    if len(arr) == 0:
        arr = np.asarray([[0, n_frames - 1]], dtype=np.int32)
    return arr

SPLIT = 'test'
ann_path  = CLIPSHOTS_ROOT/'annotations'/f'{SPLIT}.json'
list_path = CLIPSHOTS_ROOT/'video_lists'/f'{SPLIT}.txt'
VIDEO_DIR = CLIPSHOTS_ROOT/'videos'/SPLIT

with open(ann_path, encoding='utf-8') as f:
    annotations = json.load(f)
with open(list_path, encoding='utf-8') as f:
    listed = [ln.strip() for ln in f if ln.strip()]

gt_scenes = {}
skip_ann = skip_vid = skip_nf = 0
for fn in listed:
    a = annotations.get(fn)
    if a is None:
        skip_ann += 1; continue
    if not (VIDEO_DIR/fn).exists():
        skip_vid += 1; continue
    if not a.get('frame_num'):
        skip_nf += 1; continue
    stem = Path(fn).stem
    gt_scenes[stem] = transitions_to_scenes(a.get('transitions', []), int(float(a['frame_num'])))

GT_PATH = WORK/'clipshots_test_gt_scenes.pickle'
with open(GT_PATH, 'wb') as f:
    pickle.dump(gt_scenes, f, protocol=pickle.HIGHEST_PROTOCOL)
print(f'ClipShots {SPLIT}: list={len(listed)} GT={len(gt_scenes)} '
      f'skip(no_ann={skip_ann}, no_video={skip_vid}, no_frames={skip_nf})')
print('GT_PATH:', GT_PATH)


In [ ]:
# Inference + eval theo chuan goc AutoShot (mAP_f1_p_fix_r, max F1)
import numpy as np, pickle
from autoshotv2 import runtime
from autoshotv2.eval import run_video_inference
from autoshotv2.common import clean_key

# ── Ham eval goc (copy tu AutoShot_origin/utils.py) ──────────────────────────
def predictions_to_scenes(predictions):
    scenes=[]; t=t_prev=start=0; t=-1
    for i,t in enumerate(predictions):
        if t_prev==1 and t==0: start=i
        if t_prev==0 and t==1 and i!=0: scenes.append([start,i])
        t_prev=t
    if t==0: scenes.append([start,i])
    if len(scenes)==0: return np.array([[0,len(predictions)-1]],dtype=np.int32)
    return np.array(scenes,dtype=np.int32)

def evaluate_scenes(gt_scenes, pred_scenes, tol=2):
    shift=tol/2
    gt=gt_scenes.astype(np.float32)+np.array([[-0.5+shift,0.5-shift]])
    pr=pred_scenes.astype(np.float32)+np.array([[-0.5+shift,0.5-shift]])
    gt_t=np.stack([gt[:-1,1],gt[1:,0]],1); pr_t=np.stack([pr[:-1,1],pr[1:,0]],1)
    i=j=tp=fp=fn=0
    while i<len(gt_t) or j<len(pr_t):
        if j==len(pr_t): fn+=1; i+=1
        elif i==len(gt_t): fp+=1; j+=1
        elif pr_t[j,1]<gt_t[i,0]: fp+=1; j+=1
        elif pr_t[j,0]>gt_t[i,1]: fn+=1; i+=1
        else: tp+=1; i+=1; j+=1
    return tp,fp,fn

def mAP_f1_maxf1(pred_dict, gt_dict):
    thrs=np.array([0.02,0.06,0.1,0.15,0.2,0.21,0.22,0.23,0.24,0.25,0.255,0.26,0.265,0.27,
        0.275,0.28,0.2833,0.2867,0.29,0.292,0.294,0.296,0.298,0.3,0.302,0.304,0.306,0.308,
        0.31,0.3133,0.3167,0.32,0.325,0.33,0.335,0.34,0.345,0.35,0.36,0.37,0.38,0.39,0.4,
        0.5,0.6,0.7,0.8,0.9])
    best=(0,0,0,0)
    for thr in thrs:
        tp=fp=fn=0
        for name,pred in pred_dict.items():
            ps=predictions_to_scenes((pred>thr).astype(np.uint8))
            a,b,c=evaluate_scenes(gt_dict[name],ps); tp+=a; fp+=b; fn+=c
        p=tp/(tp+fp) if tp+fp else 0
        r=tp/(tp+fn) if tp+fn else 0
        f1=2*p*r/(p+r) if p+r else 0
        if f1>best[0]: best=(f1,p,r,thr)
    return best

# ── Postprocess config tu checkpoint ─────────────────────────────────────────
cfg = runtime.load_checkpoint_config(CKPT)
temperature = float(cfg.get('temperature', runtime.DEFAULT_TEMPERATURE))
sigma = float(cfg.get('sigma', runtime.DEFAULT_SIGMA))
print(f'Postprocess: temperature={temperature:.5f} sigma={sigma:.2f}')

with open(GT_PATH,'rb') as f:
    gt = pickle.load(f)
include_keys = {clean_key(k) for k in gt}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
logits = run_video_inference(
    CKPT, VIDEO_DIR, WORK/'clipshots_test_logits.pkl',
    device=device, include_keys=include_keys, resume=True,
)

pred = {clean_key(k): np.asarray(
            runtime.logits_to_probabilities(v, temperature=temperature, sigma=sigma)
        ).squeeze() for k,v in logits.items()}
gt_clean = {clean_key(k): np.asarray(v) for k,v in gt.items()}
common = sorted(set(pred)&set(gt_clean))
print(f'Videos eval: {len(common)}/{len(gt)}')

f1,p,r,thr = mAP_f1_maxf1({k:pred[k] for k in common},{k:gt_clean[k] for k in common})
print('\n=== KET QUA CLIPSHOTS TEST (CHUAN TAC GIA AutoShot) ===')
print(f'AutoShotV2 heatmap+smoothing: F1={f1:.4f}  P={p:.4f}  R={r:.4f}  thr={thr:.4f}')
print(f'(eval tren {len(common)} video ClipShots test, max-F1 sweep threshold)')
print('\nGhi chu: dien so baseline tu dung paper ban trich dan (TransNetV2 / AutoShot)')
print('de so sanh - khong hardcode o day de tranh sai lech trong bao cao.')

(WORK/'clipshots_test_result.txt').write_text(
    f'ClipShots test: F1={f1:.4f} P={p:.4f} R={r:.4f} thr={thr:.4f} (n={len(common)})\n',
    encoding='utf-8')
print('\nSaved ->', WORK/'clipshots_test_result.txt')
